In [ ]:
# ======================================================================
# STUDENT 1 - BUILDING ENERGY (BER), DUBLIN
# ======================================================================
# Research question (RQ1, from the proposal):
#   What are the dwelling characteristics that explain standardised energy
#   demand, and what energy profiles can be seen in BER-assessed homes?
#
# Parts of this notebook:
#   PART 1  Data understanding (EDA)
#   PART 2  Train / test split by time (train 2021-2024, test 2025)
#   PART 3  Data mining 1: K-Prototypes clustering (energy profiles)
#   PART 4  Data mining 2: FAMD (main dimensions of the housing stock)
#   PART 5  Regression: XGBoost -> target BerRating (kWh/m2/yr)
#   PART 6  Classification: Random Forest -> target is_B3_or_better
#   PART 7  Saving results for the group model
#
# Rules used throughout (from the module):
#   - every model starts with a real baseline (LinearRegression /
#     LogisticRegression), never a dummy or random model
#   - all preprocessing inside a Pipeline, fitted on training data only
#   - several error measures, cross-validation and bootstrap intervals
#   - explainability with permutation importance and SHAP
#
# Data: just_dublin_dataset\Just_dublin_clean_dataset\ber\ber_dublin_clean.csv
# ======================================================================

In [ ]:
# Libraries that are not in a standard install (only needed once)

%pip install xgboost shap kmodes prince

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, median_absolute_error,
                             r2_score, f1_score, precision_score, recall_score, accuracy_score,
                             cohen_kappa_score, average_precision_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             adjusted_rand_score)
from xgboost import XGBRegressor
from kmodes.kprototypes import KPrototypes
import prince
import shap

pd.set_option('display.max_columns', None)

# Fixed seed so every run gives the same results (reproducibility)
SEED = 42
np.random.seed(SEED)

In [ ]:
base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ber_file   = base + r'\just_dublin_dataset\Just_dublin_clean_dataset\ber\ber_dublin_clean.csv'
out_folder = base + r'\just_dublin_dataset\model_outputs\student1'

os.makedirs(out_folder, exist_ok=True)

In [ ]:
ber = pd.read_csv(ber_file, low_memory=False)

print(ber.shape)
ber.head(3)

In [ ]:
# Which columns are what.
# Location and date columns are kept for splitting and for checking errors
# by district; dublin_district is also used as an input.
# The targets are left out of the inputs, and so are EnergyRating and
# rating_scale, because they are just BerRating cut into letter bands
# (using them would give the answer away).

id_and_date = ['ber_id', 'nation', 'province', 'county', 'county_id', 'local_authority',
               'district_id', 'ber_location', 'DateOfAssessment', 'year', 'quarter']

targets = ['BerRating', 'EnergyRating', 'rating_scale', 'is_B3_or_better']

features = []
for col in ber.columns:
    if col not in id_and_date and col not in targets:
        features.append(col)

categorical_features = []
numeric_features = []
for col in features:
    if ber[col].dtype == 'object' or str(ber[col].dtype) == 'str':
        categorical_features.append(col)
    else:
        numeric_features.append(col)

print(len(features), 'input features')
print(len(numeric_features), 'numeric:', numeric_features)
print()
print(len(categorical_features), 'categorical:', categorical_features)

In [ ]:
# ======================================================================
# PART 1 - DATA UNDERSTANDING (EDA)
# ======================================================================
# The regression target: BerRating = energy use in kWh per m2 per year.
# Lower is better. The plot is cut at 800 so the main shape is visible.

print(ber['BerRating'].describe().round(1))

plt.figure(figsize=(10, 4))
ber['BerRating'].clip(upper=800).hist(bins=80)
plt.title('BerRating (kWh/m2/yr), Dublin 2021-2025 (values above 800 shown at 800)')
plt.xlabel('BerRating')
plt.show()

In [ ]:
# Outliers in the target, using the IQR rule (as in the class EDA notebook).
# They are NOT removed: very high values are real (small, old, badly
# insulated homes) and negative values are homes that make more energy
# than they use. RMSE will be affected by them, MAE much less.

q1 = ber['BerRating'].quantile(0.25)
q3 = ber['BerRating'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

print('Upper IQR limit:', round(upper, 1))
print('Homes above it:', (ber['BerRating'] > upper).sum(), '(', round((ber['BerRating'] > upper).mean() * 100, 1), '% )')
print('Homes below 0: ', (ber['BerRating'] < 0).sum())

In [ ]:
# The classification target: is the home B3 or better?
# About half the homes are, so the classes are fairly balanced.

print(ber['is_B3_or_better'].value_counts())
print()
print(ber['is_B3_or_better'].value_counts(normalize=True).round(3))

plt.figure(figsize=(10, 4))
ber['EnergyRating'].value_counts().sort_index().plot(kind='bar')
plt.title('Energy rating bands, Dublin')
plt.show()

In [ ]:
# Missing values in the inputs (% of rows).
# These are filled inside the model Pipeline, on training data only.

missing = (ber[features].isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Energy use by dwelling type

order = ber.groupby('DwellingType')['BerRating'].median().sort_values().index

plt.figure(figsize=(10, 5))
sns.boxplot(data=ber, y='DwellingType', x='BerRating', order=order, showfliers=False)
plt.title('BerRating by dwelling type (outliers hidden)')
plt.show()

In [ ]:
# Energy use by year of construction (newer homes should use much less)

ber['build_decade'] = (ber['Year_of_Construction'] // 10) * 10
decade = ber[ber['build_decade'] >= 1900].groupby('build_decade')['BerRating'].median()

plt.figure(figsize=(10, 4))
decade.plot(marker='o')
plt.title('Median BerRating by decade built')
plt.ylabel('BerRating')
plt.show()

ber = ber.drop(columns=['build_decade'])

In [ ]:
# Energy use and share of B3-or-better homes by Dublin district

district_summary = ber.groupby('dublin_district').agg(
    homes=('ber_id', 'count'),
    median_ber=('BerRating', 'median'),
    pct_b3_or_better=('is_B3_or_better', 'mean'))
district_summary['pct_b3_or_better'] = (district_summary['pct_b3_or_better'] * 100).round(1)
district_summary = district_summary.sort_values('median_ber')

district_summary

In [ ]:
# Correlation of the numeric inputs with BerRating (Spearman, because the
# relationships are not straight lines and there are outliers)

corr = ber[numeric_features + ['BerRating']].corr(method='spearman')['BerRating'].drop('BerRating')
corr = corr.sort_values()

plt.figure(figsize=(8, 10))
corr.plot(kind='barh')
plt.title('Spearman correlation with BerRating')
plt.show()

In [ ]:
# ======================================================================
# PART 2 - TRAIN / TEST SPLIT BY TIME
# ======================================================================
# As in the proposal: train on 2021-2024, test on 2025 (out-of-time).
# The rows are not shuffled across years, so the model never learns from
# the future.

train = ber[ber['year'] <= 2024]
test = ber[ber['year'] == 2025]

print('Train:', train.shape, ' years', train['year'].min(), '-', train['year'].max())
print('Test: ', test.shape, ' year ', test['year'].min())
print()
print('B3 or better, train:', round(train['is_B3_or_better'].mean(), 3))
print('B3 or better, test: ', round(test['is_B3_or_better'].mean(), 3))

In [ ]:
X_train = train[features]
X_test = test[features]

y_train_reg = train['BerRating']
y_test_reg = test['BerRating']

y_train_clf = train['is_B3_or_better']
y_test_clf = test['is_B3_or_better']

In [ ]:
# Preprocessing for the linear baseline models (fitted inside the Pipeline):
#   numbers:    IterativeImputer (predicts missing values from the other
#               columns, instead of filling with the mean or median),
#               then RobustScaler
#   categories: missing -> its own category 'missing', then one-hot encoding

linear_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', IterativeImputer(max_iter=5, n_nearest_features=10, random_state=SEED)),
        ('scale', RobustScaler())]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

# Preprocessing for the tree models (XGBoost, Random Forest):
# trees don't need scaling and both can handle missing numbers themselves,
# so numbers pass straight through.

tree_preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

In [ ]:
# ======================================================================
# PART 3 - DATA MINING 1: K-PROTOTYPES CLUSTERING (energy profiles)
# ======================================================================
# K-Prototypes groups homes using numbers AND categories together
# (K-Means only works with numbers). The goal is to find typical kinds of
# Dublin homes ('archetypes'). BerRating is NOT used to make the clusters;
# it is only looked at afterwards to describe them.
#
# Key building characteristics used:

cluster_numeric = ['Year_of_Construction', 'TotalFloorArea', 'UValueWall', 'UValueRoof',
                   'UValueWindow', 'HSMainSystemEfficiency']
cluster_categorical = ['DwellingType', 'MainSpaceHeatingFuel', 'MainWallType', 'VentilationMethod']
cluster_columns = cluster_numeric + cluster_categorical

In [ ]:
# K-Prototypes can't work with missing values, so only complete rows are
# used, and a random sample of 10,000 homes keeps it fast on a laptop.

complete = ber.dropna(subset=cluster_columns)
print('Homes with all cluster columns filled:', len(complete), '(', round(len(complete) / len(ber) * 100, 1), '% )')

sample = complete.sample(10000, random_state=SEED)
print('Sample size:', len(sample))

In [ ]:
# Scaling the numbers so that, for example, floor area (in m2) does not
# outweigh U-values (around 0-3). Categories stay as text.

cluster_scaler = RobustScaler()

X_cluster = sample[cluster_columns].copy()
X_cluster[cluster_numeric] = cluster_scaler.fit_transform(X_cluster[cluster_numeric])

# positions of the categorical columns (K-Prototypes needs these)
categorical_positions = list(range(len(cluster_numeric), len(cluster_columns)))
print(categorical_positions)

In [ ]:
# Elbow chart: try 2 to 8 clusters and look at the cost (lower = tighter
# clusters). The 'elbow', where the drop slows down, suggests a good number.

costs = []
k_values = range(2, 9)

for k in k_values:
    model = KPrototypes(n_clusters=k, init='Cao', n_init=1, random_state=SEED)
    model.fit(X_cluster.values, categorical=categorical_positions)
    costs.append(model.cost_)
    print('k =', k, ' cost =', round(model.cost_))

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), costs, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Cost')
plt.title('K-Prototypes elbow chart')
plt.show()

In [ ]:
# Number of clusters chosen from the elbow chart above: the cost drops a lot\n# up to 4 clusters, then only a little after that.
# (If the chart looks different on your run, change this number.)

k_chosen = 4

kproto = KPrototypes(n_clusters=k_chosen, init='Cao', n_init=1, random_state=SEED)
sample['cluster'] = kproto.fit_predict(X_cluster.values, categorical=categorical_positions)

print(sample['cluster'].value_counts().sort_index())

In [ ]:
# Stability check: run again with random starting points (init='Huang')
# and different seeds, then compare with the first result.
# Adjusted Rand Index: 1 = identical clusters, 0 = no better than chance.

for seed in [1, 2, 3]:
    other = KPrototypes(n_clusters=k_chosen, init='Huang', n_init=1, random_state=seed)
    other_labels = other.fit_predict(X_cluster.values, categorical=categorical_positions)
    print('seed', seed, ' Adjusted Rand Index =', round(adjusted_rand_score(sample['cluster'], other_labels), 3))

In [ ]:
# Describing each cluster with the original (unscaled) numbers

profile = sample.groupby('cluster')[cluster_numeric].median().round(2)
profile['homes'] = sample.groupby('cluster').size()
profile['median_BerRating'] = sample.groupby('cluster')['BerRating'].median().round(1)
profile['pct_B3_or_better'] = (sample.groupby('cluster')['is_B3_or_better'].mean() * 100).round(1)

profile

In [ ]:
# The most common category in each cluster

for col in cluster_categorical:
    print('-----', col)
    for cl in sorted(sample['cluster'].unique()):
        top = sample[sample['cluster'] == cl][col].value_counts(normalize=True)
        print('cluster', cl, ':', top.index[0], '(', round(top.iloc[0] * 100), '% )')

In [ ]:
# BerRating by cluster: the clusters were made without BerRating, so a
# clear difference here means the archetypes really capture energy
# performance.

plt.figure(figsize=(9, 4))
sns.boxplot(data=sample, x='cluster', y='BerRating', showfliers=False)
plt.title('BerRating by cluster (outliers hidden)')
plt.show()

In [ ]:
# Giving every Dublin home with complete cluster columns a cluster
# (not just the sample). Used later for cluster shares per district.

X_all = complete[cluster_columns].copy()
X_all[cluster_numeric] = cluster_scaler.transform(X_all[cluster_numeric])

complete = complete.copy()
complete['cluster'] = kproto.predict(X_all.values, categorical=categorical_positions)

print(complete['cluster'].value_counts(normalize=True).sort_index().round(3))

In [ ]:
# Share of each cluster in each district (% of homes)

cluster_share = pd.crosstab(complete['dublin_district'], complete['cluster'], normalize='index')
(cluster_share * 100).round(1)

In [ ]:
# ======================================================================
# PART 4 - DATA MINING 2: FAMD (Factor Analysis of Mixed Data)
# ======================================================================
# FAMD is like PCA but works with numbers and categories together.
# It shows the main 'directions' along which Dublin homes differ.
# Same sample and same columns as the clustering (original values; FAMD
# scales the numbers itself).

famd_data = sample[cluster_columns]

famd = prince.FAMD(n_components=5, random_state=SEED)
famd = famd.fit(famd_data)

famd.eigenvalues_summary

In [ ]:
# How much each column contributes to the first 2 dimensions.
# (Categories are split into one row per category.)

contributions = famd.column_contributions_
print('Top 10 for dimension 1:')
print(contributions[0].sort_values(ascending=False).head(10).round(3))
print()
print('Top 10 for dimension 2:')
print(contributions[1].sort_values(ascending=False).head(10).round(3))

In [ ]:
# Homes on the first two dimensions, coloured by B3 or better

coords = famd.row_coordinates(famd_data)

plt.figure(figsize=(8, 6))
plt.scatter(coords[0], coords[1], c=sample['is_B3_or_better'], cmap='coolwarm', s=4, alpha=0.5)
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('FAMD: homes on the first two dimensions (red = B3 or better)')
plt.show()

In [ ]:
# Same plot coloured by cluster, to see how the two methods agree

plt.figure(figsize=(8, 6))
plt.scatter(coords[0], coords[1], c=sample['cluster'], cmap='tab10', s=4, alpha=0.5)
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('FAMD: homes coloured by K-Prototypes cluster')
plt.show()

In [ ]:
# ======================================================================
# PART 5 - REGRESSION: predicting BerRating
# ======================================================================
# Error measures used (lower is better, except R2):
#   MAE   average error in kWh/m2/yr (main measure, not pulled by outliers)
#   RMSE  punishes big errors more
#   MedAE the typical error (half of homes have a smaller error)
#   R2    share of the variation explained
# MAPE is not used: some homes have a BerRating close to 0 or below 0
# (solar panels), which makes percentage errors explode and meaningless.

def regression_scores(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': root_mean_squared_error(y_true, y_pred),
            'MedAE': median_absolute_error(y_true, y_pred),
            'R2': r2_score(y_true, y_pred)}

regression_scoring = {'mae': 'neg_mean_absolute_error',
                      'rmse': 'neg_root_mean_squared_error',
                      'medae': 'neg_median_absolute_error',
                      'r2': 'r2'}

# 5-fold cross-validation on the training years, rows kept in date order
cv_reg = KFold(n_splits=5)

In [ ]:
# BASELINE: Linear Regression (a real, simple model, as used in class)

baseline_reg = Pipeline([
    ('prep', linear_preprocessor),
    ('model', LinearRegression())
])

cv_base = cross_validate(baseline_reg, X_train, y_train_reg, cv=cv_reg, scoring=regression_scoring, n_jobs=-1)

print('Baseline Linear Regression, 5-fold CV on 2021-2024:')
for name in ['mae', 'rmse', 'medae', 'r2']:
    scores = cv_base['test_' + name]
    if name != 'r2':
        scores = -scores
    print(f'  {name.upper():5s} {scores.mean():8.2f}  (+/- {scores.std():.2f})')

In [ ]:
# Tuning XGBoost with a small grid search.
# To keep it quick, the search runs on a random 30,000-home part of the
# training data with 3-fold CV; the best settings are then used on all of it.

xgb_pipe = Pipeline([
    ('prep', tree_preprocessor),
    ('model', XGBRegressor(n_estimators=400, tree_method='hist', random_state=SEED, n_jobs=-1))
])

param_grid = {'model__max_depth': [4, 6, 8],
              'model__learning_rate': [0.05, 0.1]}

tune_rows = X_train.sample(30000, random_state=SEED).index

search = GridSearchCV(xgb_pipe, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=1)
search.fit(X_train.loc[tune_rows], y_train_reg.loc[tune_rows])

print('Best settings:', search.best_params_)
print('Best CV MAE:  ', round(-search.best_score_, 2))

In [ ]:
# Grid search results for all settings tried

results = pd.DataFrame(search.cv_results_)[['param_model__max_depth', 'param_model__learning_rate',
                                             'mean_test_score', 'std_test_score']]
results['mean_test_score'] = -results['mean_test_score']
results.sort_values('mean_test_score').round(2)

In [ ]:
# MAIN MODEL: XGBoost with the best settings, 5-fold CV on all training data

xgb_model = search.best_estimator_

cv_xgb = cross_validate(xgb_model, X_train, y_train_reg, cv=cv_reg, scoring=regression_scoring, n_jobs=1)

print('XGBoost, 5-fold CV on 2021-2024:')
for name in ['mae', 'rmse', 'medae', 'r2']:
    scores = cv_xgb['test_' + name]
    if name != 'r2':
        scores = -scores
    print(f'  {name.upper():5s} {scores.mean():8.2f}  (+/- {scores.std():.2f})')

In [ ]:
# Final test on 2025: fit both models on all of 2021-2024, predict 2025

baseline_reg.fit(X_train, y_train_reg)
xgb_model.fit(X_train, y_train_reg)

pred_base_reg = baseline_reg.predict(X_test)
pred_xgb = xgb_model.predict(X_test)

regression_results = pd.DataFrame([regression_scores(y_test_reg, pred_base_reg),
                                   regression_scores(y_test_reg, pred_xgb)],
                                  index=['Baseline: Linear Regression', 'XGBoost']).round(2)
regression_results

In [ ]:
# Is XGBoost really better, or could it be luck? Bootstrap on the 2025 homes:
# resample the test homes 1,000 times and each time compare the two MAEs.
# (Same idea as the bootstrap notebook from class.)

errors_base = np.abs(y_test_reg.values - pred_base_reg)
errors_xgb = np.abs(y_test_reg.values - pred_xgb)

rng = np.random.default_rng(SEED)
differences = []
for i in range(1000):
    rows = rng.integers(0, len(errors_base), len(errors_base))
    differences.append(errors_base[rows].mean() - errors_xgb[rows].mean())

low, high = np.percentile(differences, [2.5, 97.5])
print('MAE improvement of XGBoost over the baseline:', round(errors_base.mean() - errors_xgb.mean(), 2), 'kWh/m2/yr')
print('95% bootstrap interval:', round(low, 2), 'to', round(high, 2))

In [ ]:
# Predicted vs actual on 2025 (sample of 3,000 homes for a readable plot)

plot_rows = rng.choice(len(y_test_reg), 3000, replace=False)

plt.figure(figsize=(6, 6))
plt.scatter(y_test_reg.values[plot_rows], pred_xgb[plot_rows], s=4, alpha=0.4)
plt.plot([0, 800], [0, 800], color='red')
plt.xlim(-50, 800)
plt.ylim(-50, 800)
plt.xlabel('Actual BerRating')
plt.ylabel('Predicted BerRating')
plt.title('XGBoost on 2025 homes')
plt.show()

In [ ]:
# Errors by Dublin district on 2025: where does the model do well or badly?

district_errors = pd.DataFrame({'district': test['dublin_district'].values,
                                'error_baseline': errors_base,
                                'error_xgb': errors_xgb})

district_mae = district_errors.groupby('district')[['error_baseline', 'error_xgb']].mean().round(1)
district_mae['homes'] = district_errors.groupby('district').size()
district_mae.sort_values('error_xgb')

In [ ]:
# Errors by size of BerRating: is the model worse for very inefficient homes?

bands = pd.cut(y_test_reg, bins=[-1000, 50, 100, 150, 200, 300, 450, 5000])
band_mae = pd.DataFrame({'band': bands.values, 'error_xgb': errors_xgb}).groupby('band')['error_xgb'].agg(['mean', 'count'])
band_mae.round(1)

In [ ]:
# EXPLAINABILITY 1: permutation importance (as in the class EDA notebook).
# Each input is shuffled in turn; the bigger the rise in error, the more the
# model relies on it. Done on 3,000 test homes to keep it quick.

perm_rows = X_test.sample(3000, random_state=SEED).index

perm = permutation_importance(xgb_model, X_test.loc[perm_rows], y_test_reg.loc[perm_rows],
                              scoring='neg_mean_absolute_error', n_repeats=5, random_state=SEED, n_jobs=1)

perm_table = pd.DataFrame({'feature': features,
                           'importance': perm.importances_mean,
                           'std': perm.importances_std}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 7))
plt.barh(perm_table['feature'].head(15)[::-1], perm_table['importance'].head(15)[::-1])
plt.xlabel('Rise in MAE when shuffled')
plt.title('XGBoost: permutation importance (top 15)')
plt.show()

perm_table.head(15).round(2)

In [ ]:
# EXPLAINABILITY 2: SHAP values (TreeExplainer for XGBoost).
# SHAP shows how much each input pushes a prediction up or down.
# 1,000 test homes are enough for the summary plots.

prep = xgb_model.named_steps['prep']
booster = xgb_model.named_steps['model']

shap_rows = X_test.sample(1000, random_state=SEED)
shap_X = prep.transform(shap_rows)
if hasattr(shap_X, 'toarray'):
    shap_X = shap_X.toarray()

feature_names = prep.get_feature_names_out()
feature_names = [name.replace('num__', '').replace('cat__', '') for name in feature_names]
shap_X = pd.DataFrame(shap_X, columns=feature_names)

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(shap_X)

shap.summary_plot(shap_values, shap_X, plot_type='bar', max_display=15)

In [ ]:
# SHAP beeswarm: red = high value of the input, blue = low value.
# E.g. a high wall U-value (red) on the right means poor walls push
# energy use up.

shap.summary_plot(shap_values, shap_X, max_display=15)

In [ ]:
# ======================================================================
# PART 6 - CLASSIFICATION: is the home B3 or better?
# ======================================================================
# Error measures used (higher is better):
#   F1 (macro)  balance of precision and recall, averaged over both classes
#   Precision, Recall (macro), Accuracy
#   Cohen's Kappa  agreement beyond chance
#   PR-AUC      area under the precision-recall curve (for class 1)

def classification_scores(y_true, y_pred, y_prob):
    return {'F1_macro': f1_score(y_true, y_pred, average='macro'),
            'Precision_macro': precision_score(y_true, y_pred, average='macro'),
            'Recall_macro': recall_score(y_true, y_pred, average='macro'),
            'Accuracy': accuracy_score(y_true, y_pred),
            'Cohen_Kappa': cohen_kappa_score(y_true, y_pred),
            'PR_AUC': average_precision_score(y_true, y_prob)}

classification_scoring = {'f1': 'f1_macro', 'precision': 'precision_macro',
                          'recall': 'recall_macro', 'accuracy': 'accuracy'}

# Stratified 5-fold CV keeps the same share of B3 homes in every fold
cv_clf = StratifiedKFold(n_splits=5)

In [ ]:
# BASELINE: Logistic Regression with balanced class weights (as in class)

baseline_clf = Pipeline([
    ('prep', linear_preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])

cv_base_clf = cross_validate(baseline_clf, X_train, y_train_clf, cv=cv_clf, scoring=classification_scoring, n_jobs=-1)

print('Baseline Logistic Regression, 5-fold CV on 2021-2024:')
for name in ['f1', 'precision', 'recall', 'accuracy']:
    scores = cv_base_clf['test_' + name]
    print(f'  {name:9s} {scores.mean():.3f}  (+/- {scores.std():.3f})')

In [ ]:
# Tuning the Random Forest with a small grid search on 30,000 training homes

rf_pipe = Pipeline([
    ('prep', tree_preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                     random_state=SEED, n_jobs=-1))
])

rf_grid = {'model__max_depth': [None, 20],
           'model__min_samples_leaf': [1, 5]}

rf_search = GridSearchCV(rf_pipe, rf_grid, cv=3, scoring='f1_macro', n_jobs=1)
rf_search.fit(X_train.loc[tune_rows], y_train_clf.loc[tune_rows])

print('Best settings:', rf_search.best_params_)
print('Best CV F1 (macro):', round(rf_search.best_score_, 3))

In [ ]:
# MAIN MODEL: Random Forest with the best settings, 5-fold CV on all training data

rf_model = rf_search.best_estimator_

cv_rf = cross_validate(rf_model, X_train, y_train_clf, cv=cv_clf, scoring=classification_scoring, n_jobs=1)

print('Random Forest, 5-fold CV on 2021-2024:')
for name in ['f1', 'precision', 'recall', 'accuracy']:
    scores = cv_rf['test_' + name]
    print(f'  {name:9s} {scores.mean():.3f}  (+/- {scores.std():.3f})')

In [ ]:
# Final test on 2025

baseline_clf.fit(X_train, y_train_clf)
rf_model.fit(X_train, y_train_clf)

pred_base_clf = baseline_clf.predict(X_test)
prob_base_clf = baseline_clf.predict_proba(X_test)[:, 1]

pred_rf = rf_model.predict(X_test)
prob_rf = rf_model.predict_proba(X_test)[:, 1]

classification_results = pd.DataFrame([classification_scores(y_test_clf, pred_base_clf, prob_base_clf),
                                       classification_scores(y_test_clf, pred_rf, prob_rf)],
                                      index=['Baseline: Logistic Regression', 'Random Forest']).round(3)
classification_results

In [ ]:
# Per-class results and confusion matrix for the Random Forest on 2025

print(classification_report(y_test_clf, pred_rf, target_names=['Worse than B3', 'B3 or better']))

ConfusionMatrixDisplay(confusion_matrix(y_test_clf, pred_rf),
                       display_labels=['Worse than B3', 'B3 or better']).plot(cmap='Blues')
plt.title('Random Forest on 2025 homes')
plt.show()

In [ ]:
# Bootstrap on the 2025 homes: how sure are we that the Random Forest
# beats the baseline on F1 (macro)?

y_true = y_test_clf.values
differences = []
for i in range(1000):
    rows = rng.integers(0, len(y_true), len(y_true))
    f1_rf = f1_score(y_true[rows], pred_rf[rows], average='macro')
    f1_base = f1_score(y_true[rows], pred_base_clf[rows], average='macro')
    differences.append(f1_rf - f1_base)

low, high = np.percentile(differences, [2.5, 97.5])
print('F1 (macro) gain of Random Forest over the baseline:',
      round(f1_score(y_true, pred_rf, average='macro') - f1_score(y_true, pred_base_clf, average='macro'), 3))
print('95% bootstrap interval:', round(low, 3), 'to', round(high, 3))

In [ ]:
# Errors by district on 2025 (share of homes classified correctly)

district_acc = pd.DataFrame({'district': test['dublin_district'].values,
                             'correct': (pred_rf == y_true)})
district_acc = district_acc.groupby('district')['correct'].agg(['mean', 'count'])
district_acc['mean'] = (district_acc['mean'] * 100).round(1)
district_acc.rename(columns={'mean': '% correct', 'count': 'homes'}).sort_values('% correct')

In [ ]:
# EXPLAINABILITY: permutation importance for the Random Forest

perm_rf = permutation_importance(rf_model, X_test.loc[perm_rows], y_test_clf.loc[perm_rows],
                                 scoring='f1_macro', n_repeats=5, random_state=SEED, n_jobs=1)

perm_rf_table = pd.DataFrame({'feature': features,
                              'importance': perm_rf.importances_mean,
                              'std': perm_rf.importances_std}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 7))
plt.barh(perm_rf_table['feature'].head(15)[::-1], perm_rf_table['importance'].head(15)[::-1])
plt.xlabel('Drop in F1 (macro) when shuffled')
plt.title('Random Forest: permutation importance (top 15)')
plt.show()

perm_rf_table.head(15).round(3)

In [ ]:
# Comparing what the two models rely on: XGBoost (energy use) and
# Random Forest (B3 or better). Similar top features = consistent story.

compare_importance = perm_table[['feature', 'importance']].rename(columns={'importance': 'XGBoost (MAE rise)'})
compare_importance = compare_importance.merge(
    perm_rf_table[['feature', 'importance']].rename(columns={'importance': 'Random Forest (F1 drop)'}), on='feature')
compare_importance.head(15).round(3)

In [ ]:
# ======================================================================
# PART 7 - SAVING RESULTS
# ======================================================================
# Results tables (for the report) and the cluster of every home
# (used by the group notebook for cluster shares per district and quarter).

regression_results.to_csv(out_folder + r'\regression_results_2025.csv')
classification_results.to_csv(out_folder + r'\classification_results_2025.csv')
profile.to_csv(out_folder + r'\cluster_profiles.csv')
perm_table.to_csv(out_folder + r'\xgboost_permutation_importance.csv', index=False)
perm_rf_table.to_csv(out_folder + r'\random_forest_permutation_importance.csv', index=False)

complete[['ber_id', 'dublin_district', 'quarter', 'cluster']].to_csv(out_folder + r'\ber_clusters.csv', index=False)

print('Saved results to', out_folder)
print(os.listdir(out_folder))